### This notebook is for doing an EDA of the American Community Survey data available from the US Census bureau. This data utlizes an API to pull information. I learned how to use their API here: https://www.census.gov/data/academy/courses/intro-to-the-census-bureau-data-api.html#1

Pulling in requests and pandas:

In [1]:
import requests

import pandas as pd

Pulling in and cleaning up the zip code list that I will use in the API url. This was sourced from Louisville Metro's Open Data portal: https://louisville-metro-opendata-lojic.hub.arcgis.com/datasets/LOJIC::jefferson-county-ky-zip-codes/explore?location=38.192334%2C-85.695891%2C10

In [2]:
ziplist_df = pd.read_csv('../data/Jefferson_County_KY_ZIP_Codes.csv')

ziplist_df.head()

,OBJECTID,ZIPCODE,SHAPEAREA,SHAPELEN
0,1,40203,8.608996e+07,76402.584655
1,2,40210,8.825244e+07,58504.651674
2,3,40059,3.342632e+08,102937.557822
3,4,40220,2.150630e+08,85234.904721
4,5,40041,3.513355e+06,8445.222310


In [3]:
ziplist_df = ziplist_df.drop('OBJECTID', axis=1)
ziplist_df = ziplist_df.drop('SHAPEAREA', axis=1)
ziplist_df = ziplist_df.drop('SHAPELEN', axis=1)

ziplist_df

,ZIPCODE
0,40203
1,40210
2,40059
3,40220
4,40041
5,40047
6,40242
7,40211
8,40215
9,40205


In [4]:
ziplist_df.dtypes

ZIPCODE    int64
dtype: object

In [5]:
ziplist_df['ZIPCODE'] = ziplist_df['ZIPCODE'].astype(str)
ziplist_df.dtypes

ZIPCODE    str
dtype: object

In [6]:
duplicate_counts = ziplist_df.apply(lambda x: x.duplicated(keep=False).sum())
print(duplicate_counts)

ZIPCODE    2
dtype: int64


In [7]:
ziplist_df[ziplist_df.duplicated(['ZIPCODE'], keep=False)]

,ZIPCODE
6,40242
18,40242


I have no need to keep two of the same zip code as I am not using the shape data, so I will drop the extra zip code.

In [8]:
cleaned_ziplist_df = ziplist_df.drop_duplicates()
cleaned_ziplist_df

,ZIPCODE
0,40203
1,40210
2,40059
3,40220
4,40041
5,40047
6,40242
7,40211
8,40215
9,40205


This next step prepares the zip codes for being added to the API's url. I am doing this to fully capture income information for Louisville Metro/Jefferson County.

In [9]:
zip_code_list = cleaned_ziplist_df['ZIPCODE'].tolist()

In [10]:
zip_codes = ','.join(zip_code_list)
zip_codes

'40203,40210,40059,40220,40041,40047,40242,40211,40215,40205,40218,40222,40223,40067,40177,40212,40214,40299,40245,40243,40025,40109,40241,40206,40219,40258,40213,40204,40208,40216,40118,40202,40217,40207,40225,40229,40023,40272,40209,40228,40291'

#### API for census.gov

The API url is below. If you want to get your own key, you can go to https://www.census.gov/data/developers/data-sets.html and click on "Request a Key" on the left side. It's free and comes to your email within a few minutes. Otherwise, you can reach out to me. Once you have the API key, fill it in under the *census_key* variable. That being said, you can run this project without obtaining an API key as long as you are under 500 requests for your IP address.

The year is a variable since I will be pulling information from 2021 to 2024. I am only looking for the median income levels, so I used the specific variable to just get that from the tables instead of all the other income information available.

In [ ]:
census_key = ''

year = '2024'

zip_code_base_url = f"https://api.census.gov/data/{year}/acs/acs5/subject?get=NAME,S1901_C01_012E&for=zip%20code%20tabulation%20area:{zip_codes}&key={census_key}"

In [12]:
zip_response = requests.get(zip_code_base_url)

zip_response.status_code

200

In [36]:
data = zip_response.json()
data

[['NAME', 'S1901_C01_012E', 'zip code tabulation area'],
 ['ZCTA5 40023', '144635', '40023'],
 ['ZCTA5 40025', '250001', '40025'],
 ['ZCTA5 40041', '153214', '40041'],
 ['ZCTA5 40047', '99124', '40047'],
 ['ZCTA5 40059', '152727', '40059'],
 ['ZCTA5 40067', '97684', '40067'],
 ['ZCTA5 40109', '75495', '40109'],
 ['ZCTA5 40118', '59316', '40118'],
 ['ZCTA5 40177', '50577', '40177'],
 ['ZCTA5 40202', '35584', '40202'],
 ['ZCTA5 40203', '30794', '40203'],
 ['ZCTA5 40204', '71116', '40204'],
 ['ZCTA5 40205', '99966', '40205'],
 ['ZCTA5 40206', '75798', '40206'],
 ['ZCTA5 40207', '101898', '40207'],
 ['ZCTA5 40208', '39560', '40208'],
 ['ZCTA5 40209', '-666666666', '40209'],
 ['ZCTA5 40210', '35347', '40210'],
 ['ZCTA5 40211', '31749', '40211'],
 ['ZCTA5 40212', '35753', '40212'],
 ['ZCTA5 40213', '55897', '40213'],
 ['ZCTA5 40214', '54931', '40214'],
 ['ZCTA5 40215', '43725', '40215'],
 ['ZCTA5 40216', '56441', '40216'],
 ['ZCTA5 40217', '69114', '40217'],
 ['ZCTA5 40218', '58811', '40218'

In [57]:

def get_median_income_data(year, zip_code_sing):
    median_income_dict = []
    zip_response = requests.get(f"https://api.census.gov/data/{year}/acs/acs5/subject?get=NAME,S1901_C01_012E&for=zip%20code%20tabulation%20area:{zip_code_sing}&key={census_key}")
    zip_response.status_code
    if zip_response.status_code == 200:
        median_income_data = zip_response.json()
        zip_code_median_income = median_income_data[1]
        print(zip_code_median_income)
        rtn_get_median_income = median_income_dict.append(zip_code_median_income)
        return rtn_get_median_income

    else: print(f"Error fetching data from census.gov: {zip_response.status_code}")

In [58]:
get_median_income_data(2024, 40206)

['ZCTA5 40206', '75798', '40206']


Now that I've got a working function to pull information, I am going to write another one to pull all information I need for median income across zip codes and years. I will add a column for year so that I know which year each median income and zip code is coming from.

In [ ]:

years = range(2021, 2025)
def get_median_income_data_all_years(years, zip_code_list):
    median_income_data_all_years = []
    for year in years:
        print(year)
        for zip_code_sing in zip_code_list:
            print(zip_code_sing)
            rtn_get_median_income = get_median_income_data(year, zip_code_sing)
            median_income_data_all_years.append({
                'year': year, 
                'zip_code': zip_code_sing, 
                'result': rtn_get_median_income
            })
    median_income_data_all_years_df = pd.DataFrame(median_income_data_all_years)
    return median_income_data_all_years_df

        


AttributeError: 'range' object has no attribute 'to_string'

In [65]:
median_income_df = get_median_income_data_all_years(years, zip_code_list)
median_income_df.head()

2021
40203
['ZCTA5 40203', '26968', '40203']
40210
['ZCTA5 40210', '29733', '40210']
40059
['ZCTA5 40059', '161079', '40059']
40220
['ZCTA5 40220', '68598', '40220']
40041
['ZCTA5 40041', '-666666666', '40041']
40047
['ZCTA5 40047', '82621', '40047']
40242
['ZCTA5 40242', '76534', '40242']
40211
['ZCTA5 40211', '29772', '40211']
40215
['ZCTA5 40215', '37535', '40215']
40205
['ZCTA5 40205', '88033', '40205']
40218
['ZCTA5 40218', '52395', '40218']
40222
['ZCTA5 40222', '73549', '40222']
40223
['ZCTA5 40223', '91647', '40223']
40067
['ZCTA5 40067', '93942', '40067']
40177
['ZCTA5 40177', '47670', '40177']
40212
['ZCTA5 40212', '31377', '40212']
40214
['ZCTA5 40214', '49427', '40214']
40299
['ZCTA5 40299', '79637', '40299']
40245
['ZCTA5 40245', '111250', '40245']
40243
['ZCTA5 40243', '79037', '40243']
40025
['ZCTA5 40025', '250001', '40025']
40109
['ZCTA5 40109', '64225', '40109']
40241
['ZCTA5 40241', '88147', '40241']
40206
['ZCTA5 40206', '62915', '40206']
40219
['ZCTA5 40219', '5094

,year,zip_code,result
0,2021,40203,None
1,2021,40210,None
2,2021,40059,None
3,2021,40220,None
4,2021,40041,None
